In [38]:
import pandas as pd
import numpy as np

In [39]:
# CSV filename
csv_file = "HiMCM 2025 - Processed Data (2)"  # <-- CHANGE THIS

# weights MUST sum to 1
weights = {
    "CO2 emissions (Tons)": 0.4,
    "Energy": 0.3,
    "Waste": 0.2,
    "Water": 0.1
}

In [40]:
df = pd.read_csv("HiMCM 2025 - Processed Data (2).csv")

# Clean 'CO2' column
df["CO2"] = df["CO2"].astype(str).str.replace("net 0", "0", regex=False) # Replace 'net 0' with '0'
df["CO2"] = df["CO2"].astype(str).str.replace(",", "", regex=False) # Remove commas
df["CO2"] = pd.to_numeric(df["CO2"], errors='coerce') # Convert to numeric, coercing errors

# Clean 'Energy' column
df["Energy"] = df["Energy"].astype(str).str.replace(",", "", regex=False) # Remove commas
df["Energy"] = pd.to_numeric(df["Energy"], errors='coerce') # Ensure numeric

# Clean 'Waste' column
df["Waste"] = df["Waste"].astype(str).str.replace("%", "", regex=False) # Remove percentage sign
df["Waste"] = pd.to_numeric(df["Waste"], errors='coerce') # Convert to numeric

# Clean 'Water' column
df["Water"] = pd.to_numeric(df["Water"], errors='coerce') # Ensure numeric

# Fill NaN values with the mean of their respective columns
for col in ["CO2", "Energy", "Waste", "Water"]:
    if df[col].isnull().any(): # Check if there are any NaN values in the column
        mean_value = df[col].mean()
        df[col] = df[col].fillna(mean_value)


required_cols = ["CO2", "Energy", "Waste", "Water"]
assert all(col in df.columns for col in required_cols), \
    f"CSV must contain: {required_cols}"

df.head()

,Superbowl (#),CO2,Energy,Waste,Water
0,"(2025) — Caesars Superdome, New Orleans, Louis...",2550000,12000,4.0,680416.68
1,"(2024) — Allegiant Stadium, Paradise (Las Vega...",0,11800,7.0,704000.00
2,"(2023) — State Farm Stadium, Glendale, Arizona",2035000,11500,7.4,643800.00
3,"(2022) — SoFi Stadium, Inglewood, California",2100000,11000,10.0,680416.68
4,"(2021) — Raymond James Stadium, Tampa, Florida",2000000,11000,12.0,661230.00


In [41]:
def normalize(series):
    return np.clip((series - series.min()) / (series.max() - series.min()), 0, 1)


In [42]:
df["n_CO2"] = normalize(df["CO2"])
df["n_Energy"] = normalize(df["Energy"])
df["n_Waste"] = normalize(df["Waste"])
df["n_Water"] = normalize(df["Water"])


In [43]:
df["CO2_score"]   = 1 - df["n_CO2"]
df["Energy_score"] = 1 - df["n_Energy"]
df["Waste_score"] = 1 - df["n_Waste"]
df["Water_score"] = 1 - df["n_Water"]


In [44]:
df["Final_Score"] = 100 * (
    weights["CO2 emissions (Tons)"] * df["CO2_score"] +
    weights["Energy"]  * df["Energy_score"] +
    weights["Waste"]   * df["Waste_score"] +
    weights["Water"]   * df["Water_score"]
)

In [45]:
# Temporary cell to inspect raw 'Energy' column
temp_df = pd.read_csv("HiMCM 2025 - Processed Data (2).csv")
print("Raw 'Energy' column sample (first 10 unique non-numeric values):")
non_numeric_energy = temp_df['Energy'][pd.to_numeric(temp_df['Energy'], errors='coerce').isna()]
print(non_numeric_energy.head(10).unique())

Raw 'Energy' column sample (first 10 unique non-numeric values):
['12,000' '11,800' '11,500' '11,000' '11,200' '10,800' '10,500' '10,400'
 '10,200']


In [46]:
df[["CO2", "Energy", "Waste", "Water", "Final_Score"]]

,CO2,Energy,Waste,Water,Final_Score
0,2550000,12000,4.0,680416.68,26.934523
1,0,11800,7.0,704000.00,65.625142
2,2035000,11500,7.4,643800.00,39.467386
3,2100000,11000,10.0,680416.68,37.847079
4,2000000,11000,12.0,661230.00,40.346623
5,1965000,11200,15.0,657282.00,39.483819
6,1892000,10800,18.0,697000.00,39.201542
7,0,10500,20.0,666585.00,72.164637
8,1938000,10400,22.0,705540.00,39.059818
9,1720000,10200,25.0,679500.00,44.714568


In [50]:
superbowl_col = [col for col in df.columns if 'Superbowl' in col][0]
df.sort_values(by='Final_Score', ascending=False)[[superbowl_col, "CO2", "Energy", "Waste", "Water", "Final_Score"]]

,Superbowl (#),CO2,Energy,Waste,Water,Final_Score
7,"(2018) — U.S. Bank Stadium, Minneapolis, Minne...",0,10500,20.0,666585.00,72.164637
24,"(2001) — Raymond James Stadium, Tampa, Florida",490000,6500,90.0,661230.00,70.165902
25,"(2000) — Georgia Dome, Atlanta, Georgia",508000,6200,95.0,680416.68,68.965895
22,"(2003) — Qualcomm Stadium, San Diego, California",517000,7000,80.0,680416.68,67.983491
18,(2007) — Dolphin Stadium (now Hard Rock Stadiu...,630000,8000,60.0,657282.00,67.086610
20,(2005) — Alltel Stadium (now EverBank Stadium)...,544000,7500,70.0,683924.00,66.921034
19,"(2006) — Ford Field, Detroit, Michigan",689000,7800,65.0,655000.00,66.259701
17,(2008) — University of Phoenix Stadium (now St...,758000,8200,55.0,643800.00,66.106185
21,"(2004) — Reliant Stadium (now NRG Stadium), Ho...",534000,7200,75.0,705540.00,65.986720
23,(2002) — Louisiana Superdome (now Caesars Supe...,503000,6800,85.0,712456.00,65.850158
